# Práctica 3: SISTEMAS BASADOS EN REGLAS

## 1. Inicialización del entorno CLIPS

In [ ]:
#!pip install clipspy
import sys
!{sys.executable} -m pip install clipspy

In [2]:
#Inicialización del entorno
import clips
import logging

     
def print_rules():
    """
    Inspecciona y muestra en pantalla todas las reglas (defrules) que han sido 
    cargadas y compilaron con éxito en el entorno de CLIPS.
    """
    for r in env.rules():
        print(r)
        
def print_agenda():
    """
    Muestra el estado actual de la Agenda de CLIPS.
    La Agenda contiene las 'activaciones', es decir, aquellas reglas cuyas condiciones 
    (lado izquierdo / LHS) se han satisfecho con los hechos actuales de la memoria de trabajo 
    y que están a la espera de ser ejecutadas al llamar a env.run().
    """
    # env.activations() devuelve la lista de reglas listas para dispararse (junto con los hechos que las activaron)
    for a in env.activations():
        print(a)
        
def print_facts():
    """
    Imprime todos los hechos (facts) almacenados actualmente en la memoria de trabajo (Working Memory).
    Para cada hecho, muestra su índice numérico asignado por CLIPS (f.index) y su contenido/estructura.
    """
    for f in env.facts():
        print(f.index, f)

def crear_entorno():
    """
    Inicializa un nuevo entorno de trabajo independiente en CLIPS, configura el sistema
    de logs de Python y redirige las salidas por pantalla internas de CLIPS (salidas estándar y de error)
    hacia el logger de Python.
    
    Retorna:
        env (clips.Environment): El objeto del entorno de CLIPS listo para ser usado.
    """
    env = clips.Environment()
    logging.basicConfig(level = 10, format = '%(message)s')
    router = clips.LoggingRouter()
    env.add_router(router)
    return env

In [2]:
env = crear_entorno()

env.define_function(print_facts)
env.define_function(print_agenda)

## 2. Descripción completa del problema a resolver

Un robot tiene que suministrar bebidas a las peticiones que se hacen de **6 puestos diferentes**. Para ello el robot utiliza un carro que es parte inseparable de su anatomía y que tiene una capacidad para **3 botellas**. 

Existe una vía que comunica consecutivamente los puestos 1,2,3,4,5 y 6 y que se extiende, además, del puesto 6 al almacén y del almacén al puesto 1, tal y como se indica en la figura. 

        Almacén ----> 1 ----> 2 ----> 3 
            |                         |
            6 <----------5 <--------- 4

**Especificaciones de operación**:

1. **Desplazamiento**: El robot tiene que desplazarse físicamente hasta el puesto donde va a atender la solicitud; sólo puede desplazarse en el sentido que indican las flechas y avanzando un solo puesto en cada acción de movimiento.
2. **Reabastecimiento**: En el almacén el robot puede llenar el carro de bebidas; supondremos que el robot siempre repone existencias hasta completar su capacidad de 3 botellas.
3. **Condiciones del pedido**: El número de botellas que solicita cada puesto nunca será mayor que 3 y el robot debe suministrar el total de botellas solicitado por un puesto en una sola entrega
4. **Estado inicial**: El robot se encuentra en el almacén con el carro de bebidas vacío y desde uno o varios puestos se solicitan un número de botellas.
5. **EStado final/objetivo**: La situación final ha de ser que todas las peticiones de bebidas hayan quedado satisfechas, el robot debe estar de nuevo en el almacén y con el carro de bebidas lleno (3 botellas). 

 Se pide diseñar un sistema basado en reglas con la sintaxis de CLIPS.

## Paso 1: Diseñar la BH: ¿Qué hechos vamos a necesitar?
Identificar y definir los hechos necesarios en CLIPS (utilizando bloques deffacts) para representar la memoria de trabajo:

1. Dónde está el robot.
2. Cómo se comunican los puestos y el almacén (conectividad del mapa).
3. Cuántas botellas lleva el robot en el carro.
4. Qué pedidos/solicitudes hay pendientes en los puestos.
5. Hechos auxiliares o de control necesarios.

### **Para describir los hechos:**

<pre style="background-color: #f5f5f5; border: 1px solid #ccc; padding: 10px; border-radius: 4px;color:#000000; width: 500px; height: 400px; overflow: auto;">
init = 
"""
(deffacts inicio
    ( ... )
    ( ... )
)
"""

Por ejemplo:
    
init = """
(deffacts inicio 
    (carretera ciuA ciuB)
    (localizacion pepe ciuA)
)
"""
</pre>
### 
**Para compilar los hechos del comando `deffacts`, ejecutar** `env.build(init)`

**Para ejecutar el motor de inferencia (cargar hechos en la BH e instancias de reglas en la agenda), ejecutar** `env.reset()`

**Para ver el estado de la BH, ejecutar** `print_facts()`

## Paso 2: Diseñar las reglas: ¿Qué acciones puede realizar el robot?

Definir las reglas de producción en CLIPS que modelan las acciones que puede realizar el robot:

1. Moverse: Avanzar de un puesto/almacén al siguiente en la vía.
2. Cargar: Reponer botellas hasta el máximo (3) únicamente estando en el almacén.
3. Servir: Entregar las botellas en un puesto si el robot está en dicho puesto y dispone de botellas suficientes.

### **Para definir las reglas:**

<pre style="background-color: #f5f5f5; border: 1px solid #ccc; padding: 10px; border-radius: 4px;color:#000000; width: 500px; height: 180px; overflow: auto;">
r1 = """
(defrule regla-mover
  ( ... )
  =>
  ( ... )
)
"""
</pre>

### 
**Para compilar las reglas definidas en los comandos `defrule`, ejecutar** `env.build(r1)`

**Para ejecutar el motor de inferencia (cargar hechos en la BH e instancias de reglas en la agenda), ejecutar** `env.reset()`

**Para ver el estado de la BH, ejecutar** `print_facts()`

**Para ver el estado de la agenda ejecutar** `print_agenda()`

## Paso 3: Probar las reglas

Para cada regla:
- Preparar un estado en la BH en el que la regla examinada debería aplicarse.
- Ejecutar la regla.
- Observar que el estado resultante coincide con el esperado.

El comando `env.reset()` lanza el motor de inferencia y se realiza el primer pattern-matching, introduciendo en la agenda las reglas instanciadas.

Con el comando `print_facts()` puedes comprobar el estado de la BH; con el comando `print_agenda()` puedes ver las reglas instanciadas en la agenda tras la primera iteración del proceso de pattern-matching. 

Con el comando `env.run(1)`, el motor de inferencia ejecuta la RHS de la primera instancia de la agenda y actualiza la BH según los cambios indicados en la RHS. Nada más terminar la ejecución de la regla, se realiza la siguiente iteración del algoritmo, es decir, se realiza el siguiente proceso de pattern-matching. Ejecutando, de nuevo,  `env.run(1)`, se ejecutará la RHS de la primera instancia de la agenda y se modificará la BH de acuerdo a las instrucciones de la RHS de la instancia de regla. Y así sucesivamente.

## Paso 4: Definir el estado objetivo y ejecutar el problema completamente

- Diseñar una regla final con mayor prioridad (`salience`) donde se refleje la siguiente situación:  todas las peticiones de bebidas hayan quedado satisfechas, 
el robot debe estar de nuevo en el almacén y con el carro de bebidas lleno.
- Probar la ejecución completa del sistema sobre el escenario de prueba inicial utilizando `env.run()` y verificar que el motor de inferencia alcanza es estado final de forma correcta.

<pre style="background-color: #f5f5f5; border: 1px solid #ccc; padding: 10px; border-radius: 4px;color:#000000; width: 650px; height: 250px; overflow: auto;">
robj = """
(defrule objetivo
  (declare (salience 100))
  ( ... )
  =>
  (printout t "Se ha alcanzado el estado final")
  (halt)
)
"""
</pre>